In [59]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [60]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [61]:
from src.trainer import IntervalTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead, set_seed
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_IT_CONFIG as CONFIG

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)


In [ ]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="TIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        context_id=i,
    )
    interval_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

    if i < len(test_tasks) - 1:
        interval_trainer.compute_rashomon_set(test, context_id=i)

### Domain Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    interval_trainer.test(test_tasks)
    if i < len(test_tasks) - 1:
        interval_trainer.compute_rashomon_set(test)

### Class Incremental Learning

In [62]:
SEED = 9
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[0, 5], [2, 3], [4, 7], [1, 8], [6, 9]]


In [63]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    interval_trainer.test(test_tasks)
    if i < len(test_tasks) - 1:
        interval_trainer.compute_rashomon_set(test)

Training Epochs: 100%|███████████████████| 5/5 [00:05<00:00,  1.12s/it, val_loss=0.0443, val_acc=0.9822, proj=None]


Test Results: [(0.032, 0.9888), (40.5101, 0.0), (34.7139, 0.0), (34.6139, 0.0), (37.5724, 0.0)] (Avg: (29.4885, 0.1978))
Initial acc constraint violation: -0.1168 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.87
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.98,  Min acc soft=0.98


100%|███████████████████████████████| 200/200 [00:14<00:00, 13.84it/s, size=1156.45, obj=0.187, min_soft_acc=0.797]


Final bbox:  Obj=0.19,  Size=1156.45,  Min acc hard=0.85,  Min acc soft=0.85
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['32.07', '182.91', '388.02', '492.71', '612.64', '722.37', '829.50', '939.54', '1049.44', '1156.45']
Checkpoint certificates: ['0.91', '0.95', '0.87', '0.86', '0.87', '0.85', '0.85', '0.85', '0.85', '0.85']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|██████████████████████| 5/5 [00:13<00:00,  2.66s/it, val_loss=3.1999, val_acc=0.0455, proj=5]


...

Final bbox:  Obj=0.00,  Size=7.74,  Min acc hard=0.12,  Min acc soft=0.12
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['4.08', '4.66', '5.05', '5.43', '5.82', '6.21', '6.60', '6.99', '7.36', '7.74']
Checkpoint certificates: ['0.12', '0.12', '0.12', '0.12', '0.12', '0.12', '0.12', '0.12', '0.12', '0.12']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|█████████████████████| 5/5 [00:12<00:00,  2.59s/it, val_loss=34.5328, val_acc=0.0000, proj=9]


Test Results: [(0.1109, 0.969), (13.7963, 0.0), (3.823, 0.1294), (33.8616, 0.0), (37.8508, 0.0)] (Avg: (17.8885, 0.2197))


Training Epochs: 100%|█████████████████████| 5/5 [00:06<00:00,  1.36s/it, val_loss=35.9729, val_acc=0.0000, proj=0]


Test Results: [(0.1109, 0.969), (13.7963, 0.0), (3.823, 0.1294), (33.8803, 0.0), (36.7255, 0.0)] (Avg: (17.6672, 0.2197))


### Class Incremental Learning + Regulariser

In [64]:
SEED = 9
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[0, 5], [2, 3], [4, 7], [1, 8], [6, 9]]


In [65]:
unbias = UnbiasRegulariser(
    lmbd=CONFIG["unbias_lambda"],
    unbias_domain=[
        torch.zeros(1, 1, 28, 28, device="cuda"),
        torch.ones(1, 1, 28, 28, device="cuda"),
    ],
)
l2 = L2Regulariser(lmbd=CONFIG["l2_lambda"])
regulariser = MultiRegulariser([l2, unbias])

In [66]:
interval_trainer = IntervalTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        regulariser=regulariser,
    )
    interval_trainer.test(test_tasks)
    if i < len(test_tasks) - 1:
        interval_trainer.compute_rashomon_set(test)

Training Epochs: 100%|███████████████████| 5/5 [00:07<00:00,  1.44s/it, val_loss=0.0482, val_acc=0.9830, proj=None]


Test Results: [(0.0266, 0.9931), (11.2351, 0.0), (9.3988, 0.0), (9.068, 0.0), (10.2248, 0.0)] (Avg: (7.9907, 0.1986))
Initial acc constraint violation: -0.1267 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.87
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|███████████████████████████████| 200/200 [00:13<00:00, 15.10it/s, size=1194.70, obj=0.192, min_soft_acc=0.807]


Final bbox:  Obj=0.19,  Size=1194.70,  Min acc hard=0.86,  Min acc soft=0.85
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['31.83', '188.00', '329.30', '461.40', '587.62', '709.26', '833.09', '957.02', '1076.57', '1194.70']
Checkpoint certificates: ['0.92', '0.87', '0.87', '0.87', '0.87', '0.86', '0.87', '0.86', '0.86', '0.86']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|██████████████████████| 5/5 [00:14<00:00,  2.91s/it, val_loss=1.6894, val_acc=0.2634, proj=1]


...

Test Results: [(0.2387, 0.9525), (1.8017, 0.1621), (8.5194, 0.0), (8.7655, 0.0), (9.8096, 0.0)] (Avg: (5.8270, 0.2229))


Training Epochs: 100%|██████████████████████| 5/5 [00:09<00:00,  1.82s/it, val_loss=8.2929, val_acc=0.0000, proj=0]


Test Results: [(0.2387, 0.9525), (1.8017, 0.1621), (9.2327, 0.0), (8.1106, 0.0), (9.7981, 0.0)] (Avg: (5.8364, 0.2229))


Training Epochs: 100%|██████████████████████| 5/5 [00:08<00:00,  1.70s/it, val_loss=8.9250, val_acc=0.0000, proj=0]


Test Results: [(0.2387, 0.9525), (1.8016, 0.1621), (9.2395, 0.0), (8.7614, 0.0), (9.063, 0.0)] (Avg: (5.8208, 0.2229))


In [ ]:
domain_accs = []
no_domain_accs = []

for domain in [True, False]:
    for i in range(5):
        SEED = i
        train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

        model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

        unbias_domain = [
            torch.zeros(1, 1, 28, 28, device="cuda"),
            torch.ones(1, 1, 28, 28, device="cuda"),
        ] if domain else None
        unbias = UnbiasRegulariser(
            lmbd=CONFIG["unbias_lambda"],
            unbias_domain=unbias_domain,
        )
        l2 = L2Regulariser(lmbd=CONFIG["l2_lambda"])
        regulariser = MultiRegulariser([l2, unbias])

        interval_trainer = IntervalTrainer(
            model,
            checkpoint=CONFIG["checkpoint"],
            n_iters=CONFIG["n_iters"],
            min_acc_limit=CONFIG["min_acc_limit"],
            min_acc_increment=CONFIG["min_acc_increment"],
            primal_learning_rate=CONFIG["primal_learning_rate"],
            dual_learning_rate=CONFIG["dual_learning_rate"],
            projection_strategy=CONFIG["projection_strategy"],
            n_certificate_samples=CONFIG["n_certificate_samples"],
            penalty_coefficient=CONFIG["penalty_coefficient"],
            paradigm="CIL",
            seed=SEED,
        )

        for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
            interval_trainer.train(
                train,
                val,
                batch_size=CONFIG["batch_size"],
                epochs=CONFIG["epochs"],
                lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"],
                regulariser=regulariser,
            )
            if i < len(test_tasks) - 1:
                interval_trainer.compute_rashomon_set(test)
            else:
                result = interval_trainer.test(test_tasks)
                accs = [res[1] for res in result]
                avg_acc = sum(accs) / 5
                if domain:
                    domain_accs.append(avg_acc)
                else:
                    no_domain_accs.append(avg_acc)

In [ ]:
print(domain_accs)
print(no_domain_accs)